# Lab qc_02 — Bell states, CHSH and teleportation

**Skills:** `qc.entanglement`, `qc.teleportation` · ~90 min · CPU (Qiskit + qiskit-aer)

### Goals
1. Prepare all four Bell states with $H$ + CNOT.
2. Compute the CHSH value exactly and by sampling on `AerSimulator` — and see $S \approx 2\sqrt2 > 2$.
3. Teleport an arbitrary qubit (a) with **mid-circuit measurement + classical feed-forward** (`if_test`) and (b) with **deferred measurement** (controlled gates instead of classical control).

### Theory recap
Bell basis: $\lvert\Phi^\pm\rangle = \tfrac{1}{\sqrt2}(\lvert00\rangle\pm\lvert11\rangle)$, $\lvert\Psi^\pm\rangle = \tfrac{1}{\sqrt2}(\lvert01\rangle\pm\lvert10\rangle)$, all reachable as $\mathrm{CNOT}(H\otimes I)\lvert xy\rangle$.

**CHSH.** With $\pm1$ observables $A(\alpha) = \cos\alpha\,Z + \sin\alpha\,X$, on $\lvert\Phi^+\rangle$ one finds $E(\alpha,\beta) = \langle A(\alpha)\otimes A(\beta)\rangle = \cos(\alpha-\beta)$.
$$S = E(a,b) + E(a,b') + E(a',b) - E(a',b'),$$
local hidden variables give $|S|\le2$; quantum mechanics reaches Tsirelson's bound $2\sqrt2$ at $a=0, a'=\pi/2, b=\pi/4, b'=-\pi/4$.
To *measure* $A(\alpha)$ on hardware rotate its eigenbasis to $Z$: apply $R_y(-\alpha)$, then measure; $E$ = average of the parity $(-1)^{x_A\oplus x_B}$.

**Teleportation** (N&C §1.3.7). $\lvert\psi\rangle_0\lvert\Phi^+\rangle_{12}$; Alice applies CNOT$_{0\to1}$, $H_0$, measures $m_0, m_1$; Bob applies $X^{m_1}$ then $Z^{m_0}$.
**Deferred measurement principle:** replacing "measure, then classically controlled $U$" by "controlled-$U$, measure later (or never)" gives the same statistics: CNOT$_{1\to2}$ and CZ$_{0\to2}$.

Qiskit reminder: amplitudes are ordered by $2q_1 + q_0$, and counts strings print registers **right-to-left** separated by spaces (last-added register leftmost).

### Read
- Preskill, Ph219 notes, Ch. 4 — https://www.preskill.caltech.edu/ph219/
- MITx 8.370.2x (teleportation, superdense coding) — https://openlearninglibrary.mit.edu/courses/course-v1:MITx+8.370.2x+1T2018/about
- IBM Quantum Learning, *Basics of Quantum Information* — https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator

X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

## 1. Bell states
`bell_circuit(which)` for `which` in `'phi+', 'phi-', 'psi+', 'psi-'` returns a 2-qubit circuit (start $\lvert00\rangle$). Global phases do not matter.

In [ ]:
def bell_circuit(which="phi+"):
    # TODO: X gates to choose |xy>, then H on qubit 0 and CNOT 0 -> 1
    raise NotImplementedError

In [ ]:
for w in ["phi+", "phi-", "psi+", "psi-"]:
    print(w, np.round(Statevector(bell_circuit(w)).data, 4))

## 2. Exact correlations and CHSH
- `measurement_observable(angle)` = $\cos a\,Z + \sin a\,X$.
- `correlation(state, a, b)` = $\langle A(a)\otimes B(b)\rangle$ for a 2-qubit state vector, **Alice = qubit 0, Bob = qubit 1**. In Qiskit order the operator is `np.kron(B, A)`.
- `chsh_value(a0, a1, b0, b1)` on $\lvert\Phi^+\rangle$.

In [ ]:
def measurement_observable(angle):
    # TODO
    raise NotImplementedError

def correlation(state, a, b):
    # TODO: <psi| kron(B(b), A(a)) |psi>, real part, as float
    raise NotImplementedError

def chsh_value(a0, a1, b0, b1):
    # TODO
    raise NotImplementedError

OPTIMAL_ANGLES = (0.0, np.pi / 2, np.pi / 4, -np.pi / 4)

## 3. CHSH by sampling
`chsh_sampled(shots, seed)`: for each of the 4 angle pairs build the Bell circuit, apply `ry(-a, 0)`, `ry(-b, 1)`, `measure_all()`, run on `AerSimulator` and estimate $E$ from parities. Use different seeds per setting.

In [ ]:
def chsh_sampled(shots=4000, seed=7):
    # TODO
    raise NotImplementedError

In [ ]:
# Experiment: S as a function of Bob's base angle b (a=0, a'=pi/2, b'=b-pi/2)
bs = np.linspace(-np.pi, np.pi, 121)
plt.plot(bs, [chsh_value(0, np.pi/2, b, b - np.pi/2) for b in bs])
plt.axhline(2, ls="--", c="gray"); plt.axhline(-2, ls="--", c="gray")
plt.axhline(2*np.sqrt(2), ls=":", c="red")
plt.xlabel("b"); plt.ylabel("S"); plt.title("CHSH: |S| > 2 is impossible classically"); plt.show()
print("sampled S =", chsh_sampled())

## 4. Teleportation with feed-forward
`teleport_circuit(theta, phi)`: qubits `q0` (input), `q1` (Alice's half), `q2` (Bob); a classical register of 2 bits.
1. Prepare $\lvert\psi\rangle = R_z(\phi)R_y(\theta)\lvert0\rangle$ on q0 (`ry` then `rz`).
2. Make $\lvert\Phi^+\rangle$ on q1, q2.
3. CNOT 0→1, H on 0, measure q0→m[0], q1→m[1].
4. `with qc.if_test((m[1], 1)): qc.x(2)` then `with qc.if_test((m[0], 1)): qc.z(2)`.

The checker undoes the preparation on q2 and verifies it always measures 0.

In [ ]:
def prepare_input(qc, q, theta, phi):
    qc.ry(theta, q)
    qc.rz(phi, q)

def teleport_circuit(theta, phi):
    # TODO
    raise NotImplementedError

## 5. Teleportation with deferred measurement
`teleport_deferred(theta, phi)`: same, but no classical bits — replace the classically-controlled $X$, $Z$ by `cx(1, 2)` and `cz(0, 2)`. Verify with `partial_trace(Statevector(qc), [0, 1])` that Bob's qubit equals $\lvert\psi\rangle$.

In [ ]:
def teleport_deferred(theta, phi):
    # TODO
    raise NotImplementedError

In [ ]:
theta, phi = 1.1, 0.4
qc = teleport_circuit(theta, phi)
print(qc.draw())
rho_bob = partial_trace(Statevector(teleport_deferred(theta, phi)), [0, 1])
target = Statevector([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])
print("fidelity (deferred):", state_fidelity(rho_bob, target))

### Reflect (on paper)
1. Show that before Alice's measurement, Bob's reduced state is $I/2$ regardless of $\lvert\psi\rangle$ — why does this forbid faster-than-light signalling?
2. Why does teleportation not violate no-cloning?

## Grade
Run all cells above, then this one. It runs the hidden checks and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("qc_02_entanglement_teleportation", globals())